# Experiment: WOP C++ r_max Escape vs Project

This notebook compares two C++ wop_cli modes for handling trajectory crossing of the outer sphere:
- escape: terminate trajectory when it reaches or exceeds r_max.
- project: project point to sphere boundary and continue (auto r_max with --r-max 0).


In [ ]:
from __future__ import annotations

import json
import math
import shutil
import statistics
import subprocess
import time
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def locate_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "external_wop_cpp").exists():
            return candidate
    raise FileNotFoundError("Could not find workspace root with external_wop_cpp")


WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
CPP_ROOT = WORKSPACE_ROOT / "external_wop_cpp"

print(f"WORKSPACE_ROOT = {WORKSPACE_ROOT}")
print(f"CPP_ROOT       = {CPP_ROOT}")
print(f"pandas available: {pd is not None}")
print(f"matplotlib available: {plt is not None}")


WORKSPACE_ROOT = c:\Users\Master\Desktop\Data\Диплом
CPP_ROOT       = c:\Users\Master\Desktop\Data\Диплом\external_wop_cpp
pandas available: True
matplotlib available: True


## Prepare C++ CLI

Build wop_cli in external_wop_cpp/build_compare_modes.


In [ ]:
cmake_exe = shutil.which("cmake")
if cmake_exe is None:
    cmake_candidates = [
        Path("C:/Program Files/CMake/bin/cmake.exe"),
        Path("C:/Program Files (x86)/CMake/bin/cmake.exe"),
    ]
    cmake_exe = next((str(p) for p in cmake_candidates if p.exists()), None)

if cmake_exe is None:
    raise FileNotFoundError(
        "CMake is not available in PATH. Install CMake and restart Jupyter, "
        "or add cmake.exe to PATH."
    )

print(f"Using CMake: {cmake_exe}")

subprocess.run([
    cmake_exe,
    "-S",
    ".",
    "-B",
    "build_compare_modes",
    "-DCMAKE_BUILD_TYPE=Release",
    "-DWOP_BUILD_TESTS=OFF",
], cwd=CPP_ROOT, check=True)

subprocess.run([
    cmake_exe,
    "--build",
    "build_compare_modes",
    "--config",
    "Release",
], cwd=CPP_ROOT, check=True)

cpp_candidates = [
    CPP_ROOT / "build_compare_modes" / "Release" / "wop_cli.exe",
    CPP_ROOT / "build_compare_modes" / "Debug" / "wop_cli.exe",
    CPP_ROOT / "build_compare_modes" / "wop_cli.exe",
    CPP_ROOT / "build_compare_modes" / "wop_cli",
]

CPP_CLI = next((p for p in cpp_candidates if p.exists()), None)
if CPP_CLI is None:
    raise FileNotFoundError("wop_cli was not found in external_wop_cpp/build_compare_modes")

CPP_CLI


Using CMake: C:\Program Files\CMake\bin\cmake.EXE


WindowsPath('c:/Users/Master/Desktop/Data/Диплом/external_wop_cpp/build_compare_modes/Release/wop_cli.exe')

## Helpers

Single-run wrapper over wop_cli and utility assertions.


In [ ]:
EXPECTED_KEYS = {
    "x0",
    "n_total",
    "n_truncated",
    "J",
    "exact",
    "abs_error",
    "S2",
    "eps",
    "mean_steps",
}


def run_cpp_once(
    n_paths: int,
    seed: int,
    mode: str,
    x0: tuple[float, float, float] = (3.0, 0.0, 0.0),
    max_steps: int = 200_000,
    escape_r_max: float = 1e6,
    project_r_max_factor: float = 3.0,
) -> dict[str, float | int | list[float]]:
    if mode not in {"escape", "project"}:
        raise ValueError("mode must be 'escape' or 'project'")

    cmd = [
        str(CPP_CLI),
        "--example",
        "box",
        "--x0",
        f"{x0[0]} {x0[1]} {x0[2]}",
        "--n",
        str(int(n_paths)),
        "--seed",
        str(int(seed)),
        "--max-steps",
        str(int(max_steps)),
        "--r-max-mode",
        mode,
    ]

    if mode == "escape":
        cmd.extend(["--r-max", str(float(escape_r_max))])
    else:
        cmd.extend(["--r-max", "0", "--r-max-factor", str(float(project_r_max_factor))])

    cmd.append("--json")

    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=CPP_ROOT, text=True)
    elapsed = time.perf_counter() - t0

    parsed = json.loads(out)
    parsed["runtime_sec"] = float(elapsed)
    parsed["mode"] = mode
    return parsed


def assert_payload(payload: dict[str, float | int | list[float]], n_paths: int) -> None:
    missing = EXPECTED_KEYS.difference(payload.keys())
    assert not missing, f"Missing keys: {sorted(missing)}"
    assert int(payload["n_total"]) == int(n_paths), f"n_total mismatch: {payload['n_total']} vs {n_paths}"
    n_truncated = int(payload["n_truncated"])
    assert 0 <= n_truncated <= int(payload["n_total"])

    for key in ["J", "exact", "abs_error", "S2", "eps", "mean_steps", "runtime_sec"]:
        value = float(payload[key])
        assert math.isfinite(value), f"{key} is not finite: {value}"
        if key in {"abs_error", "eps", "runtime_sec"}:
            assert value >= 0.0, f"{key} must be non-negative"


def run_sanity_tests() -> None:
    n_paths = 64
    seed = 20260304

    escape_payload = run_cpp_once(n_paths=n_paths, seed=seed, mode="escape")
    project_payload = run_cpp_once(n_paths=n_paths, seed=seed, mode="project")

    assert_payload(escape_payload, n_paths=n_paths)
    assert_payload(project_payload, n_paths=n_paths)

    assert escape_payload["mode"] == "escape"
    assert project_payload["mode"] == "project"

    print("Sanity tests passed for both modes (escape, project).")
    print({
        "escape_n_truncated": int(escape_payload["n_truncated"]),
        "project_n_truncated": int(project_payload["n_truncated"]),
    })


## Run Comparison Matrix

Each (n_paths, seed) pair is evaluated in both modes.


In [ ]:
N_VALUES = [100, 1000, 10000, 100000, 1000000]
SEEDS = [314159]
MODES = ["escape", "project"]

rows: list[dict[str, float | int | str | list[float]]] = []

for n_paths in N_VALUES:
    for seed in SEEDS:
        for mode in MODES:
            payload = run_cpp_once(
                n_paths=n_paths,
                seed=seed,
                mode=mode,
            )
            assert_payload(payload, n_paths=n_paths)
            rows.append({
                "mode": str(payload["mode"]),
                "n_paths": int(n_paths),
                "seed": int(seed),
                "J": float(payload["J"]),
                "exact": float(payload["exact"]),
                "abs_error": float(payload["abs_error"]),
                "eps": float(payload["eps"]),
                "S2": float(payload["S2"]),
                "mean_steps": float(payload["mean_steps"]),
                "n_total": int(payload["n_total"]),
                "n_truncated": int(payload["n_truncated"]),
                "runtime_sec": float(payload["runtime_sec"]),
            })


Collected rows: 10


[{'mode': 'escape',
  'n_paths': 100,
  'seed': 314159,
  'J': 0.33829499466882945,
  'exact': 0.35488672049383807,
  'abs_error': 0.01659172582500862,
  'eps': 0.1217226595433431,
  'S2': 0.16462673162560684,
  'mean_steps': 39.26,
  'n_total': 100,
  'n_truncated': 57,
  'runtime_sec': 0.01409729997976683},
 {'mode': 'project',
  'n_paths': 100,
  'seed': 314159,
  'J': 0.31090223164805986,
  'exact': 0.35488672049383807,
  'abs_error': 0.043984488845778213,
  'eps': 0.11178990408840098,
  'S2': 0.13885536284548766,
  'mean_steps': 22.69,
  'n_total': 100,
  'n_truncated': 0,
  'runtime_sec': 0.007462300010956824},
 {'mode': 'escape',
  'n_paths': 1000,
  'seed': 314159,
  'J': 0.3550967761926081,
  'exact': 0.35488672049383807,
  'abs_error': 0.0002100556987700286,
  'eps': 0.039447613217541484,
  'S2': 0.17290157650675037,
  'mean_steps': 38.949,
  'n_total': 1000,
  'n_truncated': 559,
  'runtime_sec': 0.014054100000066683},
 {'mode': 'project',
  'n_paths': 1000,
  'seed': 314159

In [ ]:
df = pd.DataFrame(rows).sort_values(["n_paths", "seed", "mode"]).reset_index(drop=True)
display(df)

  

,mode,n_paths,seed,J,exact,abs_error,eps,S2,mean_steps,n_total,n_truncated,runtime_sec
0,escape,100,314159,0.338295,0.354887,0.016592,0.121723,0.164627,39.260000,100,57,0.014097
1,project,100,314159,0.310902,0.354887,0.043984,0.111790,0.138855,22.690000,100,0,0.007462
2,escape,1000,314159,0.355097,0.354887,0.000210,0.039448,0.172902,38.949000,1000,559,0.014054
3,project,1000,314159,0.343410,0.354887,0.011477,0.038443,0.164204,21.342000,1000,0,0.010821
4,escape,10000,314159,0.353044,0.354887,0.001843,0.012587,0.176024,41.246200,10000,5653,0.059891
5,project,10000,314159,0.352742,0.354887,0.002144,0.012194,0.165202,21.802800,10000,0,0.042126
6,escape,100000,314159,0.354608,0.354887,0.000279,0.003985,0.176405,41.456130,100000,56372,0.545779
7,project,100000,314159,0.354368,0.354887,0.000518,0.003844,0.164195,21.383460,100000,0,0.388758
8,escape,1000000,314159,0.355033,0.354887,0.000146,0.001261,0.176774,41.542211,1000000,563642,6.980883
9,project,1000000,314159,0.355351,0.354887,0.000465,0.001217,0.164528,21.401499,1000000,0,4.787592
